# 04 — Bayesian Return Model

## 1. Title and objective

**Objective.** Estimate expected daily stock returns from the `daily_returns` SQL table using a hierarchical Bayesian Student-t model.

This notebook frames return modeling as **uncertainty-aware historical return estimation**. It summarizes what the historical return data imply about each stock's latent average daily log return, while explicitly avoiding claims that the model predicts future prices.

## 2. Why sample average returns are noisy

Daily stock returns have low signal-to-noise ratios. A typical daily expected return is small relative to daily volatility, so the arithmetic or log-return sample average can move materially when a few large positive or negative days enter or leave the sample.

This matters because:

- daily return distributions are volatile and skewed by market shocks;
- a stock with a high historical average may simply have experienced a few unusually strong days;
- a stock with a low historical average may have been affected by a few unusually bad days; and
- ranking stocks by sample average return alone can overstate precision.

The goal is therefore not just to estimate an average, but to estimate **how uncertain** that average is.

## 3. Why Bayesian modeling is useful

Bayesian modeling is useful for this task because it returns a full posterior distribution for each unknown parameter instead of a single point estimate.

In this notebook, the posterior distribution helps us:

- **estimate uncertainty** around each stock's latent daily expected return;
- **return posterior distributions** for stock-level means, stock-level volatility, group-level mean, group-level dispersion, and tail thickness;
- **avoid overconfidence in point estimates** by reporting credible intervals and probabilities; and
- compare stocks probabilistically, for example by estimating `Pr(expected return > 0)` and `Pr(stock has the highest expected return)`.

These probabilities summarize uncertainty in historical return estimates. They should not be interpreted as guaranteed future outcomes.

## 4. Why a Student-t likelihood is used

Financial returns often have **fat tails**: extreme positive or negative returns occur more often than a normal distribution would imply. A Gaussian likelihood can therefore treat crisis days or earnings-shock days as too surprising, which can distort uncertainty estimates.

A Student-t likelihood is more robust because it has an estimated degrees-of-freedom parameter, `nu`, that controls tail thickness. Lower `nu` values allow heavier tails. In this model, `nu` is learned from the data, allowing the likelihood to adapt to the empirical extremeness of daily returns.

## Setup: imports, paths, and reproducibility settings

The path setup below allows the notebook to run from either the repository root or the `notebooks/` directory. The sampling controls are intentionally centralized so a quick exploratory run can use fewer draws, while a final analysis can increase them.

In [ ]:
from pathlib import Path
import sys

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

# Resolve the repository root whether this notebook is launched from the repo
# root or from the notebooks/ directory.
NOTEBOOK_PATH = Path.cwd().resolve()
ROOT = NOTEBOOK_PATH if (NOTEBOOK_PATH / "src").exists() else NOTEBOOK_PATH.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.bayesian_models import (
    build_hierarchical_student_t_return_model,
    posterior_predictive_check,
    posterior_probability_best_return,
    posterior_probability_positive_return,
    prepare_returns_for_bayesian_model,
    sample_model,
)
from src.config import DUCKDB_PATH, FIGURES_DIR, RAW_CSV_PATH, TRADING_DAYS_PER_YEAR
from src.sql_utils import initialize_database, query_to_df, run_sql_pipeline

RANDOM_SEED = 42
DRAWS = 1000
TUNE = 1000
CHAINS = 4
TARGET_ACCEPT = 0.90

# In this repository snapshot, the source CSV may be available either at the
# configured production path or at the repository root. Prefer the configured
# path, but fall back to the checked-in sample CSV for a self-contained run.
RAW_CSV_FOR_NOTEBOOK = RAW_CSV_PATH if RAW_CSV_PATH.exists() else ROOT / "tech_stocks.csv"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
az.style.use("arviz-whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.6f}".format)

print(f"Repository root: {ROOT}")
print(f"Raw CSV used:     {RAW_CSV_FOR_NOTEBOOK}")
print(f"DuckDB path:      {DUCKDB_PATH}")
print(f"Figures dir:      {FIGURES_DIR}")

## 5. Load `daily_returns` from DuckDB

The model uses the SQL-generated `daily_returns` table. The cell below initializes DuckDB from the raw CSV if needed and runs the versioned SQL pipeline so the notebook is reproducible from a clean checkout.

In [ ]:
con = initialize_database(raw_csv_path=RAW_CSV_FOR_NOTEBOOK, db_path=DUCKDB_PATH)
run_sql_pipeline(con)

daily_returns = query_to_df(
    con,
    """
    SELECT
        symbol,
        date,
        adj_close,
        volume,
        simple_return,
        log_return,
        dollar_volume
    FROM daily_returns
    WHERE log_return IS NOT NULL
    ORDER BY symbol, date
    """,
)

display(daily_returns.head())
print(f"Loaded {len(daily_returns):,} daily return rows across {daily_returns['symbol'].nunique()} stocks.")

## 6. Prepare data for the Bayesian model

The helper `prepare_returns_for_bayesian_model` cleans the return data, enforces valid `log_return` observations, sorts rows by stock and date, and encodes each symbol as an integer index for PyMC.

In [ ]:
returns, stock_idx, symbol_lookup, modeling_df = prepare_returns_for_bayesian_model(daily_returns)
symbols = [symbol_lookup[idx] for idx in sorted(symbol_lookup)]
n_stocks = len(symbols)

modeling_summary = (
    modeling_df.groupby("symbol")
    .agg(
        observations=("log_return", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        sample_daily_mean=("log_return", "mean"),
        sample_daily_sd=("log_return", "std"),
    )
    .reset_index()
)
modeling_summary["sample_annualized_mean"] = modeling_summary["sample_daily_mean"] * TRADING_DAYS_PER_YEAR

print(f"Prepared {returns.size:,} observations for {n_stocks} stocks: {symbols}")
display(modeling_summary)

## 7. Fit the hierarchical Student-t return model

Model structure:

- `group_mu`: market-wide average daily return level;
- `group_sigma`: cross-stock dispersion in expected daily returns;
- `stock_mu`: stock-specific latent expected daily log return, partially pooled toward `group_mu`;
- `stock_sigma`: stock-specific daily return scale; and
- `nu`: Student-t degrees of freedom, which controls tail thickness.

Partial pooling lets each stock have its own expected return while regularizing noisy stock-level estimates toward the group distribution.

In [ ]:
return_model = build_hierarchical_student_t_return_model(
    returns=returns,
    stock_idx=stock_idx,
    n_stocks=n_stocks,
)

idata = sample_model(
    return_model,
    draws=DRAWS,
    tune=TUNE,
    chains=CHAINS,
    target_accept=TARGET_ACCEPT,
)

idata

## 8. ArviZ posterior summary

The summary reports central tendency, posterior uncertainty, effective sample sizes, and convergence diagnostics for the main model parameters.

In [ ]:
summary_vars = ["group_mu", "group_sigma", "stock_mu", "stock_sigma", "nu"]
posterior_summary = az.summary(
    idata,
    var_names=summary_vars,
    hdi_prob=0.90,
    round_to=6,
)
display(posterior_summary)

## 9. Plot posterior distributions of expected daily return by stock

The posterior distributions below show uncertainty in each stock's latent expected **daily** log return. Wider distributions indicate less precise historical mean-return estimates.

In [ ]:
stock_mu_samples = (
    idata.posterior["stock_mu"]
    .assign_coords(stock=symbols)
    .stack(sample=("chain", "draw"))
    .transpose("stock", "sample")
    .values
)

stock_mu_long = pd.DataFrame(stock_mu_samples.T, columns=symbols).melt(
    var_name="symbol",
    value_name="posterior_daily_return",
)

fig, ax = plt.subplots(figsize=(11, 6))
sns.kdeplot(
    data=stock_mu_long,
    x="posterior_daily_return",
    hue="symbol",
    fill=False,
    common_norm=False,
    linewidth=2,
    ax=ax,
)
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set_title("Posterior distributions of expected daily log return")
ax.set_xlabel("Expected daily log return")
ax.set_ylabel("Posterior density")
fig.tight_layout()
posterior_daily_return_path = FIGURES_DIR / "04_posterior_daily_expected_returns.png"
fig.savefig(posterior_daily_return_path, dpi=300, bbox_inches="tight")
posterior_daily_return_path

## 10. Convert posterior daily returns to annualized returns

For interpretability, the latent daily expected returns are converted to approximate annualized expected log returns using:

$$
\text{annualized\_mu} \approx \text{stock\_mu} \times 252
$$

This linear approximation is appropriate for summarizing small daily log returns, but it remains an estimate of historical average return, not a future price forecast.

In [ ]:
annualized_mu_samples = stock_mu_samples * TRADING_DAYS_PER_YEAR
annualized_quantiles = np.quantile(annualized_mu_samples, [0.05, 0.50, 0.95], axis=1)

positive_return = posterior_probability_positive_return(idata, symbol_lookup)
best_return = posterior_probability_best_return(idata, symbol_lookup)

annualized_summary = pd.DataFrame(
    {
        "symbol": symbols,
        "posterior_annualized_return_mean": annualized_mu_samples.mean(axis=1),
        "annualized_return_ci_5": annualized_quantiles[0],
        "annualized_return_median": annualized_quantiles[1],
        "annualized_return_ci_95": annualized_quantiles[2],
    }
)

return_probability_table = (
    annualized_summary
    .merge(positive_return, on="symbol")
    .merge(best_return, on="symbol")
    .sort_values("posterior_annualized_return_mean", ascending=False)
    .reset_index(drop=True)
)

return_probability_table

## 11. Show annualized return probability table

The table combines approximate annualized expected return summaries with a 90% credible interval, the posterior probability that expected daily return is positive, and the posterior probability that each stock has the highest expected daily return in the modeled universe.

In [ ]:
styled_return_probability_table = return_probability_table.style.format(
    {
        "posterior_annualized_return_mean": "{:.2%}",
        "annualized_return_ci_5": "{:.2%}",
        "annualized_return_median": "{:.2%}",
        "annualized_return_ci_95": "{:.2%}",
        "probability_positive_return": "{:.1%}",
        "probability_best_return": "{:.1%}",
    }
)
display(styled_return_probability_table)

## 12. Plot posterior probability of positive expected return

`Pr(stock_mu > 0)` estimates how much posterior mass lies above zero for each stock's latent expected daily return.

In [ ]:
positive_plot_df = return_probability_table.sort_values("probability_positive_return", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=positive_plot_df,
    x="probability_positive_return",
    y="symbol",
    color="#4C78A8",
    ax=ax,
)
ax.axvline(0.5, color="black", linestyle="--", linewidth=1)
ax.set_xlim(0, 1)
ax.set_title("Posterior probability expected daily return is positive")
ax.set_xlabel("Pr(expected daily return > 0)")
ax.set_ylabel("Symbol")
fig.tight_layout()
positive_return_path = FIGURES_DIR / "04_probability_positive_expected_return.png"
fig.savefig(positive_return_path, dpi=300, bbox_inches="tight")
positive_return_path

## 13. Plot posterior probability each stock has the highest expected return

This plot compares stocks jointly across posterior draws. For each posterior draw, the stock with the largest `stock_mu` receives one count; counts are then converted into probabilities.

In [ ]:
best_plot_df = return_probability_table.sort_values("probability_best_return", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=best_plot_df,
    x="probability_best_return",
    y="symbol",
    color="#F58518",
    ax=ax,
)
ax.set_xlim(0, max(1.0, best_plot_df["probability_best_return"].max() * 1.1))
ax.set_title("Posterior probability each stock has the highest expected return")
ax.set_xlabel("Pr(highest expected daily return)")
ax.set_ylabel("Symbol")
fig.tight_layout()
best_return_path = FIGURES_DIR / "04_probability_best_expected_return.png"
fig.savefig(best_return_path, dpi=300, bbox_inches="tight")
best_return_path

## 14. Posterior predictive check

A posterior predictive check compares observed returns with simulated returns generated by the fitted model. The purpose is to evaluate whether the Student-t model can reproduce broad features of the historical return distribution, especially tail behavior.

In [ ]:
posterior_predictive = posterior_predictive_check(return_model, idata)

ppc_values = posterior_predictive.posterior_predictive["returns"].values
ppc_matrix = ppc_values.reshape(-1, ppc_values.shape[-1])
rng = np.random.default_rng(RANDOM_SEED)
n_ppc_lines = min(100, ppc_matrix.shape[0])
ppc_draw_idx = rng.choice(ppc_matrix.shape[0], size=n_ppc_lines, replace=False)

fig, ax = plt.subplots(figsize=(10, 5))
for draw_idx in ppc_draw_idx:
    sns.kdeplot(ppc_matrix[draw_idx], color="#9E9E9E", alpha=0.08, linewidth=0.8, ax=ax)
sns.kdeplot(returns, color="black", linewidth=2.5, label="Observed daily returns", ax=ax)
ax.set_title("Posterior predictive check for daily log returns")
ax.set_xlabel("Daily log return")
ax.set_ylabel("Density")
ax.legend()
fig.tight_layout()
ppc_path = FIGURES_DIR / "04_posterior_predictive_check.png"
fig.savefig(ppc_path, dpi=300, bbox_inches="tight")
ppc_path

## 15. Interpretation

When interpreting this model, keep the following distinctions clear:

- **Historical performance is not a future price prediction.** The model estimates latent expected daily returns implied by the observed historical `daily_returns` table. It does not forecast future prices, future returns, or trading profits.
- **High expected return can still mean high uncertainty.** A stock can have a high posterior annualized mean and a wide 90% credible interval. That combination means the historical evidence is favorable on average, but the estimate is imprecise.
- **Positive-return probabilities are uncertainty summaries, not guarantees.** `Pr(expected return > 0)` measures posterior mass above zero for the latent historical mean return.
- **Best-stock probabilities are relative and posterior-based.** `Pr(highest expected return)` answers how often a stock has the largest latent mean among this stock universe across posterior draws. It is sensitive to the selected universe and sample period.
- **Partial pooling stabilizes noisy estimates.** The hierarchical prior pulls individual `stock_mu` estimates toward the shared `group_mu`, especially when a stock's return history is noisy. This reduces overreaction to extreme sample averages while still allowing genuine stock-level differences.

A practical review should focus on stocks whose posterior annualized return means are high **and** whose credible intervals and probability metrics show enough evidence to distinguish them from the group. Stocks with wide intervals may be interesting historically, but their uncertainty should be emphasized rather than hidden.

## 16. Saved figure paths

The key visual outputs are saved under `reports/figures` for reuse in reports and dashboards.

In [ ]:
saved_figures = pd.DataFrame(
    {
        "figure": [
            "Posterior expected daily returns by stock",
            "Posterior probability of positive expected return",
            "Posterior probability of highest expected return",
            "Posterior predictive check",
        ],
        "path": [
            posterior_daily_return_path,
            positive_return_path,
            best_return_path,
            ppc_path,
        ],
    }
)
display(saved_figures)